# Tutorial · 牛津 Tutorial LLM 仿真 (v6.0)

## Persona Prompt (cell 1)

You are an Oxford tutorial fellow in **价值创造与定价 (Value Creation & Pricing)**, specializing in AI product pricing strategy, OLS econometric analysis, NPV financial evaluation, price elasticity estimation, and 天道推演 competitive sandbox simulation.

**Hard rules of engagement (Oxford tutorial pedagogy + Vygotsky ZPD co-construction):**

1. **Never give direct answers.** 你绝不直接给答案。You use Socratic questioning to lead the student to discover the answer themselves.
2. **Socratic probing.** 每轮你以一个追问结束 (end each turn with a probing question), 迫使学生反思推理链。
3. **Devil's advocate.** 你故意扮演反方, 拒绝模糊断言, 要求证据 (Harvard HBS Christensen Center 风格)。
4. **Scaffold fading.** 若学生连续 2 次答错, 降一级脚手架 (从抽象追问降到具体提示), 但仍不直接给答案。
5. **Limit frequency.** 每单元每天限 1 次完整 tutorial 会话, 防止学生依赖 LLM 替代独立思考。
6. **Reject vague claims.** 若学生说"R² 高所以模型好", 你必须追问"高 R² 是否等价于因果识别? 反例?".

---

**本单元 tutorial 主题**: 给定真实 AI API 定价数据 (OpenAI/Anthropic/Google/DeepSeek) + DeepSeek V3 训练成本 $5.576M, 学生须独立产出定价决策备忘录。本 tutorial 不替学生做决策, 只通过追问暴露其推理盲点。


## Pre-Tutorial Task (cell 2 · 强制 retrieval)

**学生必须在 tutorial 开始前提交以下三件物** (Butler 2010 retrieval practice 测试效应):

1. **OLS 解读段 (200 字)**: 给定 `sm.OLS(y, sm.add_constant(X)).fit()` 输出 R²=0.859, context_window 系数 p=0.003, provider 系数 p=0.42, 学生须独立解读"哪些变量显著驱动 AI 产品定价"。
2. **NPV 决策**: 给定 DeepSeek V3 训练成本 $5.576M, 折现率 10%, 5 年现金流, 学生须独立判断"成本加成 / 价值定价 / 渗透 / 撇脂"哪种策略 NPV 最可能为正, 并说明假设。
3. **弹性 CI 决策**: 给定弹性点估计 -0.6169, 95% CI [-1.02, -0.21], 学生须独立回答"是否应基于点估计直接涨价"。

未提交任一件 -> tutorial 拒绝开始, 返回"请先完成 pre-tutorial task"。

**理由**: Oxford tutorial 是"口头辩护"而非"讲授"。学生不带 pre-task 进 tutorial = 无辩护对象 = tutorial 失效。Pre-task 强制提取 (retrieval) 比阅读 (re-study) 提升长期保留 24 个百分点 (Butler 2010: 推断题 68% vs 重学 44%)。


In [ ]:
# cell 3 · Socratic Multi-Turn Loop (>=4 轮, 静态 if/else 模拟, 不真调 API)
# Anti-stall: 本 cell 用静态 if/else 分支模拟 LLM Socratic 追问, 不调任何外部 API。

import json
from pathlib import Path

STUDENT_MODEL_PATH = Path("student_model.json")

def load_student_model():
    if STUDENT_MODEL_PATH.exists():
        return json.loads(STUDENT_MODEL_PATH.read_text(encoding="utf-8"))
    return {
        "unit": "U-skill4-day2",
        "mastery": {"ILO1_OLS": 0.0, "ILO2_NPV": 0.0, "ILO3_elasticity_montecarlo": 0.0},
        "blind_spots": [],
        "scaffold_level": 3,  # 3=抽象追问, 2=具体提示, 1=几乎给答案, 0=exit
        "turn_count": 0,
        "last_session_date": None,
        "consecutive_failures": {"ILO1": 0, "ILO2": 0, "ILO3": 0}
    }

def save_student_model(m):
    STUDENT_MODEL_PATH.write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding="utf-8")

def socratic_turn(student_answer, ilo_key, model):
    """静态 if/else 模拟牛津 fellow 的 Socratic 追问。
    不真调 API。每轮检测 defense 失败则降一级 scaffold。
    返回: (fellow_reply, is_exit)
    """
    model["turn_count"] += 1
    ans = (student_answer or "").strip().lower()
    scaffold = model["scaffold_level"]

    # ===== ILO1: OLS 解读 =====
    if ilo_key == "ILO1":
        # 轮 1: 抽象追问 (scaffold=3)
        if model["turn_count"] == 1:
            if "0.42" in ans and ("不显著" in ans or "not significant" in ans or "p>0.05" in ans):
                model["mastery"]["ILO1_OLS"] = 0.4
                return ("你说 provider p=0.42 不显著。**但我要你反问自己**: 若 R²=0.859 仍然很高, 这说明什么? 高 R² 加上一个不显著的变量, 是否矛盾? 反例? 凭什么依据?", False)
            else:
                model["consecutive_failures"]["ILO1"] += 1
                model["scaffold_level"] = 2
                return ("你还没解读 p=0.42。**Socratic 追问**: p=0.42 与 0.05 的关系是什么? 若 p>0.05, 我们能否拒绝原假设? 为什么?", False)
        # 轮 2
        if model["turn_count"] == 2:
            if "不矛盾" in ans or "其他变量解释" in ans or "not contradictory" in ans:
                model["mastery"]["ILO1_OLS"] = 0.7
                return ("好。你说其他变量解释了方差。**Devil's advocate**: 那为何还要把 provider 放进模型? 删除它会怎样? 你如何用 statsmodels 验证? 假设你删除 provider, R² 会下降多少?", False)
            else:
                model["consecutive_failures"]["ILO1"] += 1
                model["scaffold_level"] = 1
                return ("你还在模糊。**具体提示**: R² 衡量的是整个模型的解释力, p 值衡量的是单个系数。两者可以同时成立。现在请回答: 如果 R² 高但某个 p 大, 这个变量是否还有必要留在模型里? 如何决定?", False)
        # 轮 3
        if model["turn_count"] == 3:
            if "删除" in ans or "remove" in ans or "重新拟合" in ans or "f-test" in ans or "aic" in ans or "bic" in ans:
                model["mastery"]["ILO1_OLS"] = 0.85
                return ("非常好, 你提到了模型选择。**最后追问**: 在 AI 定价决策中, 若 provider 不显著, 你是否应一律删除它? 什么情况下保留不显著变量是合理的? 反例是什么?", False)
            else:
                model["consecutive_failures"]["ILO1"] += 1
                return ("你还没说具体方法。**降级提示**: 看看 AIC/BIC 或 F-test。如何用它们比较两个嵌套模型? 给我你的推理。", False)
        # 轮 4+
        if model["turn_count"] >= 4:
            if "先验" in ans or "业务" in ans or "domain" in ans or "理论" in ans or "prior" in ans:
                model["mastery"]["ILO1_OLS"] = 0.9
                model["scaffold_level"] = 0
                return ("[EXIT ILO1] 你已掌握: R²/p/模型选择的整合推理。盲点已记录到 student_model.json。请进入 ILO2 (NPV)。", True)
            else:
                model["consecutive_failures"]["ILO1"] += 1
                return ("[WEAK_LOOP ILO1] 连续未达推理深度。退出本轮, 触发 practice.md D1 Faded 重做。", True)

    # ===== ILO2: NPV =====
    if ilo_key == "ILO2":
        if model["turn_count"] == 1:
            if "-" in ans or "负" in ans or "neg" in ans or "流出" in ans or "outflow" in ans:
                model["mastery"]["ILO2_NPV"] = 0.4
                return ("你说第 0 期是负的 (投资流出)。**Socratic 追问**: 若你写成正数会发生什么? npf.npv 的内部实现如何处理符号? 反例? 凭什么?", False)
            else:
                model["consecutive_failures"]["ILO2"] += 1
                model["scaffold_level"] = 2
                return ("你没提到符号。**追问**: DeepSeek V3 训练成本 $5.576M 是收入还是支出? 在 npf.npv 的 cashflows 列表中, 第 0 期应写 5.576 还是 -5.576? 为什么?", False)
        if model["turn_count"] == 2:
            if "双解" in ans or "无解" in ans or "no solution" in ans or "符号变化" in ans or "sign change" in ans:
                model["mastery"]["ILO2_NPV"] = 0.7
                return (**"好。**Devil's advocate**: 撇脂策略 NPV 最高但 payback 最长, 渗透策略 NPV 较低但 payback 最短。若你的公司现金流紧张只能撑 18 个月, 你选哪个? 如何用 npf.payback 验证? 假设折现率从 10% 变 15%, 决策会反转吗? 反例?", False)
            else:
                model["consecutive_failures"]["ILO2"] += 1
                model["scaffold_level"] = 1
                return ("你还在模糊。**降级提示**: IRR 是使 NPV=0 的折现率。若现金流符号多次变化 (先负后正再负), IRR 可能多个或无解。重答。", False)
        if model["turn_count"] >= 3:
            if "敏感性" in ans or "sensitivity" in ans or "场景" in ans or "scenario" in ans:
                model["mastery"]["ILO2_NPV"] = 0.9
                model["scaffold_level"] = 0
                return ("[EXIT ILO2] 你已掌握: 符号约定 + IRR 多解 + 折现率敏感性。进入 ILO3 (弹性+蒙特卡洛)。", True)
            else:
                model["consecutive_failures"]["ILO2"] += 1
                return ("[WEAK_LOOP ILO2] 退出, 触发 practice.md D2 Faded 重做。", True)

    # ===== ILO3: 弹性 + 蒙特卡洛 =====
    if ilo_key == "ILO3":
        if model["turn_count"] == 1:
            if "不能" in ans or "no" in ans or "ci" in ans or "区间" in ans or "interval" in ans:
                model["mastery"]["ILO3_elasticity_montecarlo"] = 0.4
                return ("你说不能基于点估计决策。**Socratic 追问**: 若真值是 CI 下限 -1.02 而非 -0.6169, 涨价 1% 会损失多少需求? 反过来, 若真值是上限 -0.21 呢? 这两种情况下利润如何变化?", False)
            else:
                model["consecutive_failures"]["ILO3"] += 1
                model["scaffold_level"] = 2
                return ("你说可以涨价? **Devil's advocate**: 点估计 -0.6169 是真值吗? 95% CI 是 [-1.02, -0.21], 真值有 95% 概率落在此区间。若真值是 -1.02, 涨价 1% 损失 1.02% 需求, 利润是升还是降? 凭什么依据?", False)
        if model["turn_count"] == 2:
            if "100" in ans or "少" in ans or "不够" in ans or "insufficient" in ans or "10k" in ans or "10000" in ans:
                model["mastery"]["ILO3_elasticity_montecarlo"] = 0.7
                return ("好, 你说 100 次采样不够。**追问**: 天道推演约定 10k 次蒙特卡洛。为什么是 10k 而非 1k 或 100k? 采样数与估计误差的关系是什么? 如何用 np.random.seed 保证可复现?", False)
            else:
                model["consecutive_failures"]["ILO3"] += 1
                model["scaffold_level"] = 1
                return ("降级提示: 100 次采样的标准误是 sqrt(p(1-p)/100), 若 p=0.5 则 SE=0.05, 即 5% 误差。10k 次则 SE=0.005。重答为何 10k。", False)
        if model["turn_count"] >= 3:
            if "3层" in ans or "三层" in ans or "immediate" in ans or "near" in ans or "far" in ans:
                model["mastery"]["ILO3_elasticity_montecarlo"] = 0.9
                model["scaffold_level"] = 0
                return ("[EXIT ILO3] 你已掌握: CI 决策 + 采样数 + 3 层推演树。tutorial 完成, 进入 Hattie 四级反馈。", True)
            else:
                model["consecutive_failures"]["ILO3"] += 1
                return ("[WEAK_LOOP ILO3] 退出, 触发 practice.md D3 Faded 重做 + 补充 worked example。", True)

    return ("[FALLBACK] 未识别的 ILO 或答案。请重新组织。", False)

# ===== 演示: 模拟一个学生在 ILO1 上的 4 轮 tutorial =====
model = load_student_model()
demo_answers_ILO1 = [
    "provider p=0.42 不显著, context_window p=0.003 显著",
    "不矛盾, 因为其他变量 (context_window, has_reasoning) 解释了方差, R² 是整体的",
    "删除 provider 重新拟合, 用 AIC/BIC 或 F-test 比较嵌套模型",
    "业务先验: 若行业知识认为 provider 有战略意义, 即使不显著也保留"
]
print("=" * 70)
print("DEMO: 牛津 tutorial 仿真 - ILO1 OLS 解读 (4 轮 Socratic)")
print("=" * 70)
for i, ans in enumerate(demo_answers_ILO1, 1):
    print(f"\n--- Turn {i} ---")
    print(f"[Student]: {ans}")
    reply, is_exit = socratic_turn(ans, "ILO1", model)
    print(f"[Oxford Fellow]: {reply}")
    if is_exit:
        print(f"\n[EXIT triggered at turn {i}]")
        break
save_student_model(model)
print(f"\n[student_model.json updated: mastery.ILO1_OLS = {model['mastery']['ILO1_OLS']}, scaffold_level = {model['scaffold_level']}]")


In [ ]:
# cell 4 · student_model.json 读写 (跨单元复用)
# 这个 cell 展示 student_model.json 的完整结构, Day 3 (Agent经济) 会读取本文件个性化起点。

import json
from pathlib import Path
from datetime import date

SAMPLE_STUDENT_MODEL = {
    "unit": "U-skill4-day2",
    "unit_topic": "价值创造+定价",
    "mastery": {
        "ILO1_OLS": 0.9,          # 0-1, 0.9 = 已达 mastery_threshold (>=0.8)
        "ILO2_NPV": 0.72,          # 0.72 >= 0.7 阈值
        "ILO3_elasticity_montecarlo": 0.6   # < 0.7 (能独立解阈值), 进入 weak_loop
    },
    "blind_spots": [
        {
            "ilo": "ILO3",
            "topic": "10k 蒙特卡洛采样数的必要性",
            "evidence": "tutorial turn 2 学生未主动提到 100 次采样误差",
            "remediation": "practice.md D3 Faded 重做 + schedule.json C4 首日复习"
        },
        {
            "ilo": "ILO3",
            "topic": "天道推演 3 层推演树 (immediate/near/far) 完整性",
            "evidence": "tutorial turn 3 学生只展开 immediate 层",
            "remediation": "practice.md D3 Independent 重做"
        }
    ],
    "scaffold_level": 1,        # 0=exit, 1=几乎给答案, 2=具体提示, 3=抽象追问
    "turn_count": 4,
    "last_session_date": str(date.today()),
    "consecutive_failures": {"ILO1": 0, "ILO2": 0, "ILO3": 2},
    "cross_unit_pointer": {
        "next_unit": "U-skill4-day3 (Agent经济)",
        "reused_concepts": ["天道推演 3 层推演树", "蒙特卡洛采样数约定"],
        "day3_personalization": "Day 3 tutorial 起点脚手架应降为 level 2 (具体提示), 因为本单元 ILO3 未达 mastery"
    }
}

Path("student_model.json").write_text(json.dumps(SAMPLE_STUDENT_MODEL, ensure_ascii=False, indent=2), encoding="utf-8")
loaded = json.loads(Path("student_model.json").read_text(encoding="utf-8"))
print("student_model.json written and reloaded:")
print(json.dumps(loaded, ensure_ascii=False, indent=2))


## Hattie 四级 Formative Feedback (cell 5)

> Hattie (2007, *Review of Educational Research* 77(1):81-112) 元分析显示反馈是学习效应最大的单一干预之一 (d≈0.79)。Hattie 四级反馈避免 Self 级表扬 (d≈0.09, 几乎无效), 聚焦 TASK/PROCESS/SELF-REG/FEED-FORWARD。

本单元 tutorial 结束后, 系统根据 `student_model.json` 自动生成以下四级反馈:

### [TASK] 任务级反馈 (针对本次 tutorial 的具体任务)
- **ILO1 OLS 解读**: 你在 turn 1 正确识别了 p=0.42 不显著, 但在 turn 2 一开始未主动指出"R² 高与单变量不显著不矛盾"。任务级修正: 下次遇到高 R² + 高 p 的组合, 先回答"R² 是模型整体, p 是单变量"。
- **ILO2 NPV**: 你正确指出第 0 期符号为负, 但未提到 IRR 在符号多次变化时可能多解。任务级修正: 每次算 IRR 前先检查现金流符号序列。
- **ILO3 弹性**: 你正确指出不能基于点估计决策, 但在蒙特卡洛采样数上未主动提到 10k 约定。任务级修正: 蒙特卡洛默认 10k, 这是天道推演硬约束。

### [PROCESS] 过程级反馈 (针对学生的推理策略)
- 你的推理策略在 ILO1/ILO2 上是"先报数字再解读"--这是正确的。但在 ILO3 上你跳过了"先报 CI 再决策"的中间步, 直接跳到结论。过程级建议: 强制自己在每个数值结论后追加"这对应的 CI 是什么 / 决策概率是多少"。
- 你在 devil's advocate 追问下能调整答案 (ILO1 turn 3 提出模型选择), 这是好的元认知策略。继续保持。

### [SELF-REG] 自我调节级反馈 (针对学生的元认知)
- 你在 ILO3 连续 2 次未达推理深度, 触发 weak_loop。问自己: 是知识缺口 (不知道 10k 约定) 还是策略缺口 (知道但没想起来)? 若是后者, 在 schedule.json C4 卡片的复习中加入"为何 10k"的主动回忆。
- 你的 scaffold_level 从 3 降到 1, 说明你在抽象追问下需要更多脚手架。自我调节建议: 下次 tutorial 前先完成 practice.md 的 Faded 阶段, 提升抽象追问下的应答能力。

### [FEED-FORWARD] 前馈级反馈 (针对下一次学习)
- **下一步**: 触发 practice.md D3 的 weak_loop (回退 Faded + 补充 worked example + 24h 间隔重测)。
- **跨单元前馈**: Day 3 (Agent 经济) 将复用本单元的"天道推演 3 层推演树"。你在 ILO3 上的盲点已写入 `student_model.json`, Day 3 tutorial 起点脚手架将自动降为 level 2 (具体提示), 直到你重测达标。
- **复习调度**: schedule.json 的 C3 (弹性 CI) 和 C4 (蒙特卡洛) 卡片首日复习 (due[0]=1) 必须在 24 小时内完成, 否则 FSRS-6 算法会重置间隔。

> **Hattie 注意**: 本反馈**不包含** Self 级表扬 (如"你做得很好""你很聪明")。Self 级反馈效应量 d≈0.09 几乎无效, 且可能损害成长心态 (Dweck)。所有反馈聚焦 TASK/PROCESS/SELF-REG/FEED-FORWARD。


## 限频 + Exit Artifact (cell 6)

### 限频策略 (防 LLM 依赖, Oxford tutorial 真实约束)

Oxford 真实 tutorial 是每周 1 次, 强制 1 对 1-3。本 LLM 仿真保持这一约束:

- **每单元每天限 1 次完整 tutorial 会话** (限频 / daily limit / 1次/天)。
- 触发限频: 学生在同一天第 2 次启动 tutorial -> 系统返回"今日 tutorial 已用尽。建议你先做 practice.md 的 Independent 阶段, 明日再来。间隔 24 小时是 FSRS-6 间隔重复的最小间隔。"
- **理由 (Vygotsky ZPD + 防依赖)**: 限频迫使学生先独立尝试, 而非把 LLM 当答案机。Socratic 追问的价值在于"被迫自己推理", 频繁访问会绕过这一过程。Oxford tutorial 的稀缺性正是其效力的来源。
- **例外**: weak_loop 触发的补充 worked example 不计入限频, 但必须在 24 小时后重测。

### Exit Artifact (tutorial 完成时强制产出)

每次 tutorial 结束, 学生必须提交 exit artifact, 包含:

1. **2-3 个盲点 (blind_spots)**: 从 `student_model.json` 的 `blind_spots` 字段读取。例:
   - 盲点 1: "10k 蒙特卡洛采样数的必要性"
   - 盲点 2: "天道推演 3 层推演树 (immediate/near/far) 完整性"
   - 盲点 3: "IRR 在符号多次变化时可能多解"

2. **推荐复习单元 (cross_unit_pointer)**:
   - 24h 内: schedule.json C3 + C4 首日复习 (due[0]=1)。
   - 72h 内: practice.md D3 Faded 阶段重做 (weak_loop)。
   - 跨单元: Day 3 (Agent 经济) tutorial 起点脚手架降为 level 2。

3. **自我承诺**: 学生写一句话承诺"我将在 [具体时间] 完成 [具体复习动作]", 写入 `student_model.json` 的 `commitment` 字段。

未提交 exit artifact -> tutorial 不算完成, 不更新 `student_model.json` 的 `last_session_date`, 下次启动仍计为同一天 (触发限频)。

---

*本 notebook v6.0 新增。所有 Socratic 追问回指 v5.0 既有真实库: statsmodels OLS R²=0.859 / numpy-financial NPV (DeepSeek V3 $5.576M) / scipy.stats 弹性 -0.6169 / 天道推演 10k 蒙特卡洛。本 notebook 用静态 if/else 模拟 LLM 响应, 不真调 API, 符合 anti-stall 约束。*
